In [ ]:
import glob
import re
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, gaussian_filter
from aicsimageio import AICSImage, types
from os import mkdir

from stardist.models import StarDist2D

from matplotlib.colors import LinearSegmentedColormap
from skimage.measure import regionprops

## Overview
This is a stripped down version of the `czi_microscopy` notebook to work with the Olympus. It is meant to loop through a directory of RGB \*tif files and make pngs. Ideally, there are no-probe controls taken at the same intensity, with those tif's in a separate directory, from which the `read_controls` function can estimate intensities per channel and then be passed to `layoutImg` to do the actual running.

In [ ]:
def read_controls(imgs, mq=0.95, smooth=False):
    channel_max_uint16 = np.zeros((len(imgs), 1))
    for enum,i in enumerate(imgs):
        img = AICSImage(i).data
        img = img[0,:,0,:,:]
        
        img = np.moveaxis(img, 0,2)
        #print(img.shape)
        if img.shape[-1] > 3:
            continue
        if smooth:
            for chan in range(img.shape[-1]):
                img[:,:,chan] = median_filter(img[:,:,chan], size=3)
                #img[:,:,chan] = gaussian_filter(img[:,:,chan], sigma=5)
            #img = img / img.max(axis=(0,1))
        channel_max_uint16[enum,:] = np.quantile(img, mq, axis=(0,1))
        #img = np.where(img < 0, 0, img)
        
    channel_max_uint16 = channel_max_uint16 * 1
    #print(channel_max_uint16)
    return channel_max_uint16

imgs=glob.glob("eub338_vnat_probe_test_20260122/20260122_100x_cy3Gam42_CONTROL/*tif")
#print(imgs)
controls = read_controls(imgs, mq=0.95).max(axis=0)

controls

#imgs=glob.glob("3768_2c_nodye_control/Image *.czi")#[1:2]
#controls3768 = read_controls(imgs).max(axis=0)


## Calculating per-channel max across a series of images

Quick way to generate a list of maximum values for use in `layoutImg`

#### Relevant arguments
`mq`= maximum quantile (1=highest pixel, 0.9=90th percentile). Default=0.99


In [ ]:
def quantitative_max(imgs, mq=0.99, quant_channels=[1]):
    channel_max_uint16 = np.zeros((len(imgs),len(quant_channels)))
    for enum,i in enumerate(imgs):
        img = AICSImage(i).data
        img = img[0,:,0,:,:]
        
        img = np.moveaxis(img, 0,2)
        print(img.shape)

        for chan in range(img.shape[-1]):
            img[:,:,chan] = median_filter(img[:,:,chan], size=3)
            #img[:,:,chan] = gaussian_filter(img[:,:,chan], sigma=5)
        #img = img / img.max(axis=(0,1))
        channel_max_uint16[enum,:] = np.quantile(img, mq, axis=(0,1))
        #img = np.where(img < 0, 0, img)
        
    channel_max_uint16 = channel_max_uint16 * quant_channels
    print(channel_max_uint16)
    return channel_max_uint16

#imgs=glob.glob("eub338_vnat_probe_test_20260122/20260122_100x_Al488Eub/*.tif")#[1:2]
#print(imgs)
#maxes = quantitative_max(imgs).max(axis=0)

In [ ]:
#cmapCol=np.array(((0,0.5,1),(1,0,0)))  2 col
cmapCol=np.array(((0,0.4,1),(0,1,0),(1,0,0),(1,1,0)))  #4 col

cmaps=[LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[0]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[1]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[2]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[3]], N=256)]

def layoutImg(i, mq=0.1, labs=['Eub-Al488'], cmaps=cmaps, cmapCol=cmapCol, cmapI=1,
              scale=20, pngsize=16, save=True, folder='', suf='', 
              controls=[], quantmax=[], DIC=False, pix_um=0.07, plot=True,
              stardist=False, segstats=False):
    img = AICSImage(i)
    
    img = img.data
    #print(img.shape)
    if img.shape[2] > 1:
        print('zstack?')
        if zstack:
            #print(img.shape)
            img = np.max(img, axis=2)
            img = np.expand_dims(img, 2)
            #print(img.shape)
                
    img = img[0,:,0,:,:]
    
    img = np.moveaxis(img, 0,2)
    #print(img.shape)
    if segstats:
        imgquant = img.copy()
    if len(controls) == 0:
        controls = np.repeat(0, img.shape[-1])
    #print(img.max(axis=(0,1)))
    minquant=np.quantile(img, mq, axis=(0,1))
    
    
    controls = np.where(controls == 0, minquant, controls)
    
    img = img - np.array(controls)
    maxquant = np.max(img, axis=(0,1))
    
    if DIC:
        img = img - np.quantile(img, mq, axis=(0,1))[:-1]

    img = np.where(img < 0, 0, img)
    
    if len(quantmax) == 0:
        img = img / (1+ img.max(axis=(0,1)))
    else:
        img = img / np.where(quantmax == 0, maxquant, quantmax)
    #img = np.where(img < 0, 0, img)
    
    chans = img.shape[-1]
    
    if plot:
        fig, axs=plt.subplots(nrows=1, ncols=2, figsize=(2.1*pngsize, pngsize))

        props = dict(linewidth=0, fc='black')

        axs[0].imshow(img[:,:,0], cmap=cmaps[cmapI], vmin=0, vmax=1)
        axs[0].text(0.98,0.98, labs[cmapI], fontsize=16, c=cmapCol[cmapI], ha='right', va='top',transform = axs[0].transAxes, bbox=props)

        axs[0].axvspan(xmin=.7*img.shape[0], xmax=.7*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.09, fc='w')
        axs[0].text(.7*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')

        axs[0].axis('off')
        
    if stardist:
        sdist = StarDist2D.from_pretrained('2D_versatile_fluo')
        masks, _ = sdist.predict_instances(img[:,:,0])
        if not segstats:
            if plot:
                axs[1].imshow(masks>1, cmap='gray')
                axs[1].text(0.98,0.98, 'Seg. mask', fontsize=16, c='w', ha='right', va='top',transform = axs[1].transAxes, bbox=props)
        else:
            regions = regionprops(masks)#measure.label(red_rois))
            intensity=np.zeros(len(regions))
            size=np.zeros(len(regions))
            #r=0
            for r,reg in enumerate(regions):
                pts = reg.coords
                intensity[r] = np.mean([imgquant[x,y,0] for x, y in pts])
                size[r] = reg.area
                #r+=1
            
            #print(intensity)
            if plot:
                vp = axs[1].violinplot(intensity,showmeans=False, showmedians=False, showextrema=False)
                [pc.set_facecolor(cmapCol[cmapI]) for pc in vp['bodies']]
                axs[1].plot(np.random.normal(1, 0.04, size=len(intensity)), intensity, c=cmapCol[cmapI], 
                              alpha=1, marker='.', linestyle='None')
                if len(controls) > 0:
                    axs[1].axhline(y=controls[0],c='black', linestyle='--')

                axs[1].text(0.5,0.98, 'N: ' + str(len(intensity)), 
                              fontsize=16, c=cmapCol[cmapI], ha='center', va='top',transform = axs[1].transAxes, bbox=dict(linewidth=0, fc='white'))
                axs[1].axis('on')
    else:
        intensity=[]

    if plot:
        plt.tight_layout()
    
    outf = re.sub('.[tcz][[izv][fi]$', suf+'.png', i)
    outf= '-'.join(outf.split('/')[-4:])
    outf=folder + '/' + outf
    #print(outf)
    if save:
        try:
            plt.savefig(outf)    
        except FileNotFoundError:
            mkdir(folder)
            plt.savefig(outf)
            
    return img, intensity


## Example of how it is run

#### Arguments:
`i`= path to image file

`mq`= minimum quantile (percentile), default=0.1

`labs`= labels for the channel, should be exactly as long as the number of channels (so length 1). default ['Eub-Al488'] 

`cmaps`= colormaps as list of LinearSegmentedColormaps

`cmapCol`= tuple/list of rgb values, used for font labels

`cmapI`= index of the cmap/cmapCol combination to use. Simple way to set a consistent color scheme

`pix_um`= **CRITICAL** to set this since .tif does not have it embedded. Size of pixel in microns, default 0.07.

`scale`= sclaebar in microns, default=20

`DIC`= Is this a DIC type image (should it be inverted)? Default False

 ----
`pngsize`= size of png, default 16

`save`= Save images or just run? Default True (save)

`folder`= name of folder in which to save images, will create if does not exist. Default '' (no folder)

`suf`= suffix to attach to filename before the .png, default '' (no suffix)

---------
`controls`= 1D list the length of channels, e.g. [0,0,0,0] if 4 channel image. values are the maximum control value for thresholding. 0 means to disregard controls for that channel, e.g. [0,1083,1024,5000] does not apply control to DAPI (first channel) but gives channel-specific thresholds. Default is [] which is no controls.

`quantmax`= reverse of controls, same format. This normalizes the max value of all images to a constant number (usually the brightest pixel in any imgae). default [] which is no normalization.

-------
`omnipose`= run omnipose object detection, default False (don't run)

`stardist`= run stardist object detection, default False (don't run). This is usually better than omnipose

`segstats` = run statistics per ROI. Default False. Only relevant if omnipose or stardist is True.


In [ ]:
imgs=glob.glob("15581B_SLAC/glut/20260205_15581B_slac_blut_AF488/*tif")[0:3]
#print(imgs)
#imgs=glob.glob("3768_2c_NR_WGA_4days_well3/Image 11.czi")#[1:2]

x = [layoutImg(im, mq=0.1, labs=['DAPI','Eub-Al488','Gam-cy3','Eub-Al647'], scale=10, suf='', 
               DIC=False, quantmax=[],#maxes,
               save=False,
               folder='TEMPFOLDER', plot=True,
             #controls=controls, #quantmax=maxes,
               pngsize=6, stardist=False, segstats=False)
             #pngsize=6, stardist=True, segstats=True)#, controls=controls3767)
   for im in imgs]# if not re.search('tilescan', im)]

### Quick look at intensity between negative controls and positive controls

Legacy of this specific snapshot. Only relevant if you have positive and negative controls, and you saved the stardist ROI stats as output.

In [ ]:
def signalToNoise(img_stats, cont_stats, labs=['Eub-Al488'], cmaps=cmaps, cmapCol=cmapCol, cmapI=1,
              scale=20, pngsize=16, save=True, folder='', suf='', saveas=False,
              controls=[], quantmax=[], DIC=True, pix_um=0.07, stardist=False, segstats=False):

    stats=np.concatenate([y[1] for y in img_stats])
    
    img=img_stats[np.random.randint(0,len(img_stats),1)[0]][0]
    cimg=cont_stats[np.random.randint(0,len(cont_stats),1)[0]][0]
    
    chans = img.shape[-1]
    
    fig = plt.figure(figsize=(2.1*pngsize, pngsize))
    gs = fig.add_gridspec(2,3)
    iaxs = fig.add_subplot(gs[0, :])
    caxs = fig.add_subplot(gs[1, :])
    saxs = fig.add_subplot(gs[:, 2])
    #fig, axs=plt.subplots(nrows=2, ncols=2, figsize=(2.1*pngsize, pngsize))
        
    props = dict(linewidth=0, fc='black')
    
    iaxs.imshow(img[:,:,0], cmap=cmaps[cmapI], vmin=0, vmax=1)
    iaxs.text(0.98,0.98, labs[cmapI]+' exp', fontsize=16, c=cmapCol[cmapI], ha='right', va='top',transform = iaxs.transAxes, bbox=props)
    caxs.imshow(cimg[:,:,0], cmap=cmaps[cmapI], vmin=0, vmax=1)
    caxs.text(0.98,0.98, labs[cmapI]+' cont', fontsize=16, c=cmapCol[cmapI], ha='right', va='top',transform = caxs.transAxes, bbox=props)
    
    iaxs.axvspan(xmin=.9*img.shape[0], xmax=.9*img.shape[0] + (scale/pix_um), ymin=0.07, ymax=0.09, fc='w')
    iaxs.text(.9*img.shape[0] + (scale/2/pix_um), 0.94*img.shape[0], str(scale) + ' µm', fontsize=16, c='w', ha='center', va='top')

    iaxs.axis('off')
    caxs.axis('off')
        
    vp = saxs.violinplot(stats/controls[0], showmeans=False, showmedians=False, showextrema=False)
    [pc.set_facecolor(cmapCol[cmapI]) for pc in vp['bodies']]
    saxs.plot(np.random.normal(1, 0.04, size=len(stats)), stats/controls[0], c=cmapCol[cmapI], 
                  alpha=1, marker='.', linestyle='None')
    if len(controls) > 0:
        saxs.axhline(y=np.mean(stats)/controls[0],c='black', linestyle='--')
        saxs.text(0.98,0.98, 'Mean SNR: --', fontsize=16, c='white', ha='right', va='top',transform = saxs.transAxes, bbox=props)
        saxs.text(0.98,0.93, 'SNR @ 1.5: ..', fontsize=16, c='white', ha='right', va='top',transform = saxs.transAxes, bbox=props)
    
    saxs.axhline(y=1.5,c='#808080', linestyle=':')

    saxs.set_ylabel('SNR')

    
    plt.tight_layout()
    
    if saveas:
        plt.savefig(saveas)


In [ ]:
signalToNoise(x, c, labs=['DAPI','Eub-Al488','Gam-cy3','Eub-Al647'], scale=10, 
              suf='', save=False, folder='TEMPFOLDER',
               controls=controls, #quantmax=maxes,
               pngsize=8, stardist=False, segstats=False)

In [ ]:
cmapCol=np.array(((0,0.4,1),(0,1,0),(1,0,0),(1,0.9,0)))  #4 col

cmaps=[LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[0]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[1]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[2]], N=256),
       LinearSegmentedColormap.from_list("Custom", [(0,0,0), cmapCol[3]], N=256)]


signalToNoise(x, c, labs=['DAPI','Eub-Al488','Gam-cy3','Eub-Al647'], 
              cmaps=cmaps, cmapCol=cmapCol, cmapI=ci, scale=10, 
              suf='', save=False, folder='TEMPFOLDER', pix_um=0.18,
               controls=controls, #quantmax=maxes,
              saveas='eub_benchmarking/40x_Eub-Al647.png',
               pngsize=8, stardist=False, segstats=False)